In [4]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from src.gtsrb import NUM_CLASSES, GTSRB_CLASSES, get_dataloaders, save_predictions

In [2]:
train_loader, val_loader, test_loader = get_dataloaders(img_size=32, batch_size=128)

In [3]:
def train_model(optimizer_name):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = nn.Sequential(

        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Flatten(),

        nn.Linear(64 * 8 * 8, 128),
        nn.ReLU(),

        nn.Linear(128, NUM_CLASSES)

    ).to(device)

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=0.001
        )

    elif optimizer_name == "Momentum":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=0.001,
            momentum=0.9
        )

    elif optimizer_name == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=0.001
        )

    epochs = 5

    losses = []
    accuracies = []

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss

        losses.append(epoch_loss)

        # -----------------------------
        # VALIDAÇÃO
        # -----------------------------

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                predictions = outputs.argmax(dim=1)

                correct += (
                    predictions == labels
                ).sum().item()

                total += labels.size(0)
    
        accuracy = 100 * correct / total

        accuracies.append(accuracy)

        print(
            f"{optimizer_name} | "
            f"Epoch {epoch+1} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Validation Accuracy: {accuracy:.2f}%"
        )

    return model, losses, accuracies

In [15]:
def metrics(model, test_loader, device):

    model.eval()

    cm = np.zeros((43, 43), dtype=int)

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predictions = torch.max(outputs, 1)

            for label, prediction in zip(labels, predictions):

                cm[label.item(), prediction.item()] += 1

    np.set_printoptions(threshold=np.inf)

    print("Matriz de Confusão:")
    print(cm)

    acc_class = [0 for _ in GTSRB_CLASSES]

    print("\nAcurácia por classe:")

    for i in range(43):

        total_class = np.sum(cm[i, :])

        if total_class > 0:

            acc = 100 * (cm[i, i] / total_class)

        else:

            acc = 0

        acc_class[i] = acc

        print(f"Classe {i}: {acc:.2f}%")

    worst_class = acc_class.index(min(acc_class))
    best_class = acc_class.index(max(acc_class))

    print(
        f"\nWorst class: {worst_class} "
        f"- {min(acc_class):.2f}%"
    )

    print(
        f"Best class: {best_class} "
        f"- {max(acc_class):.2f}%"
    )

    return cm, acc_class

In [ ]:
def generate_predictions(model):

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model.eval()

    predictions_list = []

    with torch.no_grad():

        for images, _ in test_loader:

            images = images.to(device)

            outputs = model(images)

            predictions = outputs.argmax(dim=1)

            predictions_list.extend(
                predictions.cpu().numpy()
            )
        metrics(model, test_loader, device)
    return predictions_list

In [26]:
model_sgd, losses_sgd, accs_sgd = train_model("SGD")

SGD | Epoch 1 | Loss: 624.3742 | Validation Accuracy: 5.86%
SGD | Epoch 2 | Loss: 616.4380 | Validation Accuracy: 6.16%
SGD | Epoch 3 | Loss: 606.4043 | Validation Accuracy: 6.10%
SGD | Epoch 4 | Loss: 595.1438 | Validation Accuracy: 6.02%
SGD | Epoch 5 | Loss: 585.6692 | Validation Accuracy: 8.22%


In [10]:
model_momentum, losses_momentum, accs_momentum = train_model("Momentum")

c:\Users\rafae\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Momentum | Epoch 1 | Loss: 597.8239 | Validation Accuracy: 13.06%
Momentum | Epoch 2 | Loss: 545.4555 | Validation Accuracy: 18.73%
Momentum | Epoch 3 | Loss: 486.1424 | Validation Accuracy: 29.95%
Momentum | Epoch 4 | Loss: 402.7751 | Validation Accuracy: 42.23%
Momentum | Epoch 5 | Loss: 316.8130 | Validation Accuracy: 54.92%


In [27]:
model_adam, losses_adam, accs_adam = train_model("Adam")

Adam | Epoch 1 | Loss: 266.7132 | Validation Accuracy: 85.74%
Adam | Epoch 2 | Loss: 48.5443 | Validation Accuracy: 95.70%
Adam | Epoch 3 | Loss: 19.7830 | Validation Accuracy: 97.15%
Adam | Epoch 4 | Loss: 11.7029 | Validation Accuracy: 97.56%
Adam | Epoch 5 | Loss: 7.7567 | Validation Accuracy: 98.52%


In [30]:
print('Metrics for SGD')
y_pred_adam = generate_predictions(model_sgd)

Metrics for SGD
Matriz de Confusão:
[[  0  26  24   0   0   0   0   0   0   0   0   0   0  10   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0]
 [  0 292  57   0   0   0   0  58   0   0   0   0  31 281   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   1   0   0   0   0]
 [  0 233 166   0   0   0   0 205   0   0   0   0  76  70   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0]
 [  0 128   4   0   0   0   0 120   0   0   0   0 135  60   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   3   0   0   0   0]
 [  0 122  57   0   0   0   0 198   0   0  10   0 213  60   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0]
 [  0 294  65   0   0   0   0 136   0   0   0   0  75  60   0   0   0

KeyboardInterrupt: 

In [ ]:
print('Metrics for Momentum')
y_pred_adam = generate_predictions(model_momentum)

Metrics for Momentum


AttributeError: 'Tensor' object has no attribute 'eval'

In [ ]:
print('Metrics for adam')
y_pred_adam = generate_predictions(model_adam)

In [ ]:
epochs_range = range(1, 6)

plt.figure(figsize=(8,5))

plt.plot(
    epochs_range,
    losses_sgd,
    marker='o',
    label='Loss'
)

plt.title('Curva de Loss')
plt.xlabel('Épocas')
plt.ylabel('Loss')

plt.xticks(epochs_range)

plt.grid(True)

plt.legend()

plt.show()


plt.figure(figsize=(8,5))

plt.plot(
    epochs_range,
    accs_sgd,
    marker='o',
    label='Validation Accuracy'
)

plt.title('Curva de Accuracy')
plt.xlabel('Épocas')
plt.ylabel('Accuracy (%)')

plt.xticks(epochs_range)

plt.grid(True)

plt.legend()

plt.show()

In [ ]:
epochs_range = range(1, 6)

plt.figure(figsize=(8,5))

plt.plot(
    epochs_range,
    losses_momentum,
    marker='o',
    label='Loss'
)

plt.title('Curva de Loss')
plt.xlabel('Épocas')
plt.ylabel('Loss')

plt.xticks(epochs_range)

plt.grid(True)

plt.legend()

plt.show()


plt.figure(figsize=(8,5))

plt.plot(
    epochs_range,
    accs_momentum,
    marker='o',
    label='Validation Accuracy'
)

plt.title('Curva de Accuracy')
plt.xlabel('Épocas')
plt.ylabel('Accuracy (%)')

plt.xticks(epochs_range)

plt.grid(True)

plt.legend()

plt.show()

In [ ]:
epochs_range = range(1, 6)

plt.figure(figsize=(8,5))

plt.plot(
    epochs_range,
    losses_adam,
    marker='o',
    label='Loss'
)

plt.title('Curva de Loss')
plt.xlabel('Épocas')
plt.ylabel('Loss')

plt.xticks(epochs_range)

plt.grid(True)

plt.legend()

plt.show()


plt.figure(figsize=(8,5))

plt.plot(
    epochs_range,
    accs_adam,
    marker='o',
    label='Validation Accuracy'
)

plt.title('Curva de Accuracy')
plt.xlabel('Épocas')
plt.ylabel('Accuracy (%)')

plt.xticks(epochs_range)

plt.grid(True)

plt.legend()

plt.show()